# 02c — External factors

## What this notebook does

Notebooks 02 / 02b collect DeFi yield data. To answer our three research questions we also need a few **external** series:

| Question | External factor needed | Source used |
|---|---|---|
| Q1 — Are ETH staking yields falling because of more staked ETH or lower fees? | Ethereum daily fees, staked-ETH proxy | DeFiLlama `overview/fees/ethereum` + Lido+RP TVL (already collected) |
| Q2 — Do stablecoin APYs move with the Fed funds rate? | Short-term risk-free rate | Yahoo Finance `^IRX` (13-week T-bill, ~Fed funds proxy) |
| Q3 — Does higher APY attract more TVL? | ETH price (to convert ETH-denominated TVL to USD context) | Yahoo Finance `ETH-USD` |

## Why these proxies

- **Fed funds rate proxy:** the FRED API requires a free API key and its CSV download was timing out, so we use `^IRX` (13-week T-bill). It tracks Fed funds almost 1-to-1 and is what short-duration stablecoin yield products are actually benchmarked against (sUSDS, sDAI are backed by short-duration treasuries).
- **Staked ETH:** beaconcha.in is behind Cloudflare; instead we use the Lido stETH + Rocket Pool rETH supply (in ETH terms) as a proxy. Lido alone has been ~28–30% of all staked ETH consistently, so its trend is highly correlated with the total.

## Outputs

- `data/processed/external_eth_price.csv` — daily ETH close in USD
- `data/processed/external_risk_free_rate.csv` — daily 13-week T-bill yield (% APY)
- `data/processed/external_eth_fees.csv` — daily Ethereum protocol fees (USD)
- `data/processed/external_staked_eth_proxy.csv` — derived Lido+RP staked ETH proxy

In [ ]:
import requests
import pandas as pd
import yfinance as yf
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW       = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_RAW.mkdir(parents=True, exist_ok=True)

# Window roughly matching our pool history (Lido goes back to mid-2022)
START = "2022-05-01"
END   = pd.Timestamp.today().strftime("%Y-%m-%d")

## 1. ETH price (USD)

Daily close from Yahoo Finance ticker `ETH-USD`.

In [ ]:
eth_raw = yf.download("ETH-USD", start=START, end=END, progress=False, auto_adjust=True)
# yfinance returns a MultiIndex (field, ticker). Flatten.
if isinstance(eth_raw.columns, pd.MultiIndex):
    eth_raw.columns = [c[0] for c in eth_raw.columns]
eth_raw.index.name = "date"
eth_price_df = eth_raw[["Close"]].rename(columns={"Close": "eth_price_usd"}).reset_index()
eth_price_df["date"] = pd.to_datetime(eth_price_df["date"]).dt.tz_localize(None)
print(f"ETH price rows: {len(eth_price_df)}, {eth_price_df['date'].min().date()} → {eth_price_df['date'].max().date()}")
eth_price_df.head()

## 2. Risk-free rate proxy (^IRX, 13-week T-bill)

`^IRX` is quoted as an annualized yield in %. It's a strong proxy for the Fed funds rate.

In [ ]:
rf_raw = yf.download("^IRX", start=START, end=END, progress=False, auto_adjust=False)
if isinstance(rf_raw.columns, pd.MultiIndex):
    rf_raw.columns = [c[0] for c in rf_raw.columns]
rf_raw.index.name = "date"
risk_free_df = rf_raw[["Close"]].rename(columns={"Close": "rf_rate_percent"}).reset_index()
risk_free_df["date"] = pd.to_datetime(risk_free_df["date"]).dt.tz_localize(None)
print(f"Risk-free rate rows: {len(risk_free_df)}, range {risk_free_df['rf_rate_percent'].min():.2f}% → {risk_free_df['rf_rate_percent'].max():.2f}%")
risk_free_df.head()

## 3. Ethereum daily fees

From DeFiLlama's chain-level fees endpoint. This is the total protocol fees paid on Ethereum each day in USD — a clean proxy for network activity / demand for blockspace.

In [ ]:
FEES_URL = "https://api.llama.fi/overview/fees/ethereum?excludeTotalDataChart=false"
r = requests.get(FEES_URL, timeout=60)
r.raise_for_status()
fees_payload = r.json()

# Save raw response for traceability
(DATA_RAW / "llama_eth_fees_chart.csv").write_text(
    "timestamp,fees_usd\n" + "\n".join(f"{t},{v}" for t,v in fees_payload["totalDataChart"])
)

eth_fees_df = pd.DataFrame(fees_payload["totalDataChart"], columns=["timestamp", "fees_usd"])
eth_fees_df["date"]      = pd.to_datetime(eth_fees_df["timestamp"], unit="s")
eth_fees_df["fees_usd"]  = pd.to_numeric(eth_fees_df["fees_usd"], errors="coerce")
eth_fees_df = eth_fees_df[["date", "fees_usd"]]
eth_fees_df = eth_fees_df[eth_fees_df["date"] >= START].reset_index(drop=True)
print(f"ETH fees rows (since {START}): {len(eth_fees_df)}")
eth_fees_df.tail()

## 4. Staked ETH proxy

We derive staked ETH from the protocol TVL data already collected by notebook 02 (`staking_tvl_timeseries.csv`).

Steps:
1. Take Lido and Rocket Pool TVL (USD).
2. Divide by the daily ETH price → staked ETH in ETH units.
3. Lido is ~28–30% of all staked ETH historically, so we scale: `total_staked_proxy ≈ lido_staked_eth / 0.28`.

The absolute scale is approximate; what we care about is the *trend* (how it grew).

In [ ]:
staking_tvl = pd.read_csv(DATA_PROCESSED / "staking_tvl_timeseries.csv")
staking_tvl["date"] = pd.to_datetime(staking_tvl["date"]).dt.tz_localize(None).dt.normalize()

eth_price_daily = eth_price_df.copy()
eth_price_daily["date"] = eth_price_daily["date"].dt.normalize()

lido = staking_tvl[staking_tvl["protocol"] == "Lido"][["date", "tvl_usd"]].rename(columns={"tvl_usd": "lido_tvl_usd"})
rp   = staking_tvl[staking_tvl["protocol"] == "Rocket Pool"][["date", "tvl_usd"]].rename(columns={"tvl_usd": "rp_tvl_usd"})
lido["date"] = lido["date"].dt.normalize()
rp["date"]   = rp["date"].dt.normalize()

staked = lido.merge(rp, on="date", how="outer").merge(eth_price_daily, on="date", how="left").sort_values("date")
# Forward-fill weekend ETH prices (Yahoo only trades workdays, crypto trades 24/7)
staked["eth_price_usd"] = staked["eth_price_usd"].ffill()

staked["lido_staked_eth"] = staked["lido_tvl_usd"] / staked["eth_price_usd"]
staked["rp_staked_eth"]   = staked["rp_tvl_usd"]   / staked["eth_price_usd"]
# Lido share is ~28% of staked ETH (varies 25–33%); use 0.28 as constant scaling factor
LIDO_SHARE_OF_STAKED = 0.28
staked["total_staked_eth_proxy"] = staked["lido_staked_eth"] / LIDO_SHARE_OF_STAKED

# Total ETH supply is ~120M; staked share = staked / supply
ETH_SUPPLY_APPROX = 120_000_000
staked["staked_share_proxy"] = staked["total_staked_eth_proxy"] / ETH_SUPPLY_APPROX

staked_eth_df = staked[["date", "lido_staked_eth", "rp_staked_eth", "total_staked_eth_proxy", "staked_share_proxy"]].copy()
print(f"Staked-ETH proxy rows: {len(staked_eth_df)}")
print("Share grew from {:.1%} to {:.1%}".format(
    staked_eth_df['staked_share_proxy'].iloc[0],
    staked_eth_df['staked_share_proxy'].iloc[-1],
))
staked_eth_df.tail()

## Save

In [ ]:
outputs = {
    "external_eth_price.csv":         eth_price_df,
    "external_risk_free_rate.csv":    risk_free_df,
    "external_eth_fees.csv":          eth_fees_df,
    "external_staked_eth_proxy.csv":  staked_eth_df,
}
for name, df in outputs.items():
    path = DATA_PROCESSED / name
    df.to_csv(path, index=False)
    print(f"  {path.name:38s} {len(df):>6,} rows")

## What we have now

Data collection is complete. We can now join these external series to the pool yields and answer:

- Q1 — regress staking APY on `staked_share_proxy` and `eth_fees`
- Q2 — regress stablecoin APY on `rf_rate_percent`
- Q3 — regress `log(TVL)` on APY and category dummies

All of this happens in notebooks 03 (explore), 04 (model), 05 (communicate).